In [1]:
import pandas as pd

In [2]:
df_ground_truth = pd.read_csv('ground_truth-new.csv')
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

# Running RAG

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [8]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes — you can still join the course even if you just found out about it. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

In [9]:
assistant.total_cost()

0.00050475

In [10]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [11]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes — you can still join the course even if you just found out about it. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [12]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [13]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [14]:
assistant.total_cost()

0.0009735

In [15]:
assistant.reset_usage()

In [16]:
assistant.total_cost()

0.0

In [17]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [19]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth[:-100], generate_rag_answer)

  0%|          | 0/620 [00:00<?, ?it/s]

In [20]:
df_results = pd.DataFrame(results)
df_results.head()

,question,answer_llm,answer_orig,document
0,Can I still join the course if I just found ou...,"Yes, you can still join the course. If you wan...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Is it too late to start this course now?,You can start whenever you want. The videos an...,"Yes, but if you want to receive a certificate,...",74eb249bbf
2,"If I join late, can I still get a certificate ...","Yes, but only if you complete the capstone pro...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,What do I need to do to be eligible for the ce...,"To be eligible for the certificate, you need t...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,Are project submissions still open for new stu...,"Yes — if the submission form is still open, ne...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [24]:
assistant.total_cost()

1.4406555000000005

In [23]:
df_results.to_csv("rag_results.csv", index=False)

# LLM AS A JUDGE

In [34]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

# AQA - Answer , Question, Answer

In [32]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [25]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [26]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [28]:
answers=results
rec = answers[0]

In [29]:
rec

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [30]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [35]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joiners can still participate, but certificate eligibility requires submitting the project before submissions close. It is semantically equivalent.', score='good')

In [37]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [38]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the full meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project before submissions close. Semantically equivalent.', score='good')

In [43]:
calc_price(usage)

{'input_cost': 0.00021975, 'output_cost': 0.0002205, 'total_cost': 0.00044025}

In [39]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/620 [00:00<?, ?it/s]

In [42]:
results[1]

({'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf',
  'score': 'bad',
  'reasoning': 'The ground truth says it is not too late to start, but adds an important condition: to receive a certificate, the project must be submitted while submissions are still being accepted. The AI answer says you can start whenever you want and mentions materials and deadlines, but it omits the certificate/submission condition. Since that is the key point of the original answer, the response is not fully equivalent.'},
 ResponseUsage(input_tokens=288, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=95, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=383))

In [45]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [46]:
df_eval = pd.DataFrame(evaluations)

In [47]:
df_eval.head()

,question,document,score,reasoning
0,Can I still join the course if I just found ou...,74eb249bbf,good,The AI answer preserves the original meaning: ...
1,Is it too late to start this course now?,74eb249bbf,bad,The ground truth says it is not too late to st...
2,"If I join late, can I still get a certificate ...",74eb249bbf,good,The AI answer preserves the main point: late j...
3,What do I need to do to be eligible for the ce...,74eb249bbf,good,The ground truth says that to receive a certif...
4,Are project submissions still open for new stu...,74eb249bbf,good,The AI answer preserves the key meaning of the...


In [48]:
calc_total_price(usages)

0.4456297500000002

In [49]:
df_eval.score.value_counts()

score
good    600
bad      20
Name: count, dtype: int64

In [50]:
df_eval.score.value_counts(normalize=True)

score
good    0.967742
bad     0.032258
Name: proportion, dtype: float64

In [51]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 600/620 = 96.77%


In [52]:
df_eval.to_csv("rag_evaluations-new.csv", index=False)